<a href="https://colab.research.google.com/github/Davidson-Open-University/phy107-labs/blob/main/notebooks/PHY107-Lab2-Reading-Instruments-and-Uncertainty.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PHY 107 Lab 2: Reading Instruments and Uncertainty

**Unit 2: Measurements and error analysis** · General Practical Physics I · Davidson Open University Global

**In this lab you will:**
- read a vernier calliper and a micrometer screw gauge, and correct for zero error;
- express uncertainty as an absolute and a percentage value;
- combine uncertainties when results are multiplied;
- find a quantity from the gradient of a straight-line graph.

**Time:** about 60 minutes in four parts:
- **Part A – Explore** (about 10 minutes): run the online lab and the examples, and read the explanations.
- **Part B – Practise** (about 20 minutes): fill in the blanks. A self-check tells you if you are right.
- **Part C – Build a tool** (about 20 minutes): complete a short analysis function and test it on data.
- **Part D – Your own data and reflection** (about 10 minutes): analyse readings you took in the online lab (or at home), and write short answers.

**How to use this notebook**
1. Run each code cell in order: click it and press **Shift + Enter** (or the ▶ button).
2. Where you see `...`, replace it with your own code, run the cell, then run the **check** cell under it.
   - ✅ means correct. ❌ gives you a hint: try again. ⏳ means you have not filled in the blank yet.
   - The checks never show the answer, so you learn by working it out. You can try as many times as you like.
3. Stuck? Read the hint, look back at the study session, or ask in the discussion forum. Do not skip ahead: each part builds on the last.
4. Save your work: in Colab choose **File → Save a copy in GitHub** and save it to your own `phy107-labs` repository. Offline in Jupyter, press **Ctrl + S** and later upload the file with **Add file → Upload files** on GitHub.

**Honest data:** keep your own readings separate from the example data, and label every simulated or supplied reading as such. Never invent readings.

*This lab is practice to help you learn and prepare for the unit assignment (lab report). It is not graded unless your tutor tells you otherwise.*

In [ ]:
#@title ▶ Run this cell first: it sets up the self-checks (you do not need to read it)
# You do not need to read or change this code.
import hashlib as _h
class _Blank(Exception): pass
def _norm(x, unordered=False, dp=6):
    try:
        import numpy as _np
        if isinstance(x, _np.ndarray): x = x.tolist()
        elif isinstance(x, _np.bool_): x = bool(x)
        elif isinstance(x, _np.integer): x = int(x)
        elif isinstance(x, _np.floating): x = float(x)
    except ImportError:
        pass
    if x is Ellipsis or x is None: raise _Blank()
    if isinstance(x, bool): return 'b:' + str(x)
    if isinstance(x, (int, float)): return 'n:' + repr(round(float(x), dp) + 0.0)
    if isinstance(x, str): return 't:' + ' '.join(x.lower().split())
    if isinstance(x, (set, frozenset)): return 's{' + ','.join(sorted(_norm(i, False, dp) for i in x)) + '}'
    if isinstance(x, (list, tuple)):
        items = [_norm(i, False, dp) for i in x]
        if unordered: items.sort()
        return 'l[' + ','.join(items) + ']'
    if isinstance(x, dict): return 'd{' + ','.join(sorted(_norm(k, False, dp) + '=' + _norm(v, False, dp) for k, v in x.items())) + '}'
    try:
        import sympy as _sp
        if isinstance(x, _sp.Basic):
            x = _sp.sympify(x)
            if x.is_number: return 'n:' + repr(round(float(x), dp) + 0.0)
            syms = sorted(x.free_symbols, key=str)
            vals = []
            for t in (0.3, 0.7, 1.3):
                v = complex(x.subs({s: t + 0.11 * i for i, s in enumerate(syms)}).evalf())
                vals.append(round(v.real, 6) + 0.0)
            return 'e:' + repr(vals)
    except ImportError:
        pass
    return 'r:' + repr(x)
def _hash(q, s): return _h.sha256(('phy107|' + q + '|' + s).encode()).hexdigest()[:16]
def check(q, value):
    """Check your answer to question q. Prints ✅, ❌ with a hint, or ⏳ if the blank is not filled yet."""
    spec = _CHECKS[q]
    try:
        if callable(value) and 'args' in spec:
            outs = []
            for a in spec['args']:
                try:
                    outs.append(value(*a))
                except _Blank:
                    raise
                except Exception as e:
                    print(f'❌ Question {q}: your function stopped with an error: {type(e).__name__}: {e}')
                    return
            if all(o is None or o is Ellipsis for o in outs): raise _Blank()
            got = _norm(outs, False, spec.get('dp', 6))
        elif spec.get('logic'):
            if value is None or value is Ellipsis: raise _Blank()
            import sympy as _sp
            from itertools import product as _p
            syms = sorted(_sp.sympify(value).free_symbols, key=str)
            got = 'L' + ''.join('1' if bool(_sp.sympify(value).subs(dict(zip(syms, v)))) else '0' for v in _p([True, False], repeat=len(syms))) + ':' + ','.join(map(str, syms))
        else:
            got = _norm(value, spec.get('unordered', False), spec.get('dp', 6))
    except _Blank:
        print(f'⏳ Question {q}: fill in the blank (replace ...) and run the cell above, then run this cell again.')
        return
    if _RECORD is not None:
        _RECORD[q] = _hash(q, got); print(f'✅ Question {q}: recorded.'); return
    if _hash(q, got) == spec['h']:
        print(f'✅ Question {q}: correct! ' + spec.get('ok', 'Well done.'))
    else:
        print(f'❌ Question {q}: not yet. Hint: ' + spec['hint'])
_RECORD = None

_CHECKS = {
    '2.1': {'h': 'ea1d4b0f64e29038', 'hint': '23 + 7 × 0.1.', 'dp': 2},
    '2.2': {'h': '8d0ffdea32f917ed', 'hint': 'A positive zero error is subtracted.', 'ok': 'Always check the zero before measuring.', 'dp': 2},
    '2.3': {'h': '941bd0756d557480', 'hint': '5.5 + 28 × 0.01.', 'ok': 'A micrometer is ten times finer than a vernier calliper.', 'dp': 2},
    '2.4': {'h': '8a9829ab40155014', 'hint': 'Divide the uncertainty by the value, then multiply by 100.', 'dp': 1},
    '2.5': {'h': 'e1d9a6ca865dd9ac', 'hint': 'A = L * W, and dA = A * (dL / L + dW / W).', 'ok': 'So A = 1009 ± 16 mm²: quote the uncertainty to 1–2 significant figures.', 'dp': 1},
    '2.6': {'h': '281b03acd4c40b2e', 'hint': 'gradient = np.polyfit(L_data, T_data**2, 1)[0]; then g = 4 * np.pi**2 / gradient.', 'ok': 'A graph uses all the readings at once, so it gives a more reliable g than one reading.', 'dp': 1},
    '2.7': {'h': '88d6b5e722e48666', 'hint': 'return round(abs(uncertainty / value) * 100, 2)', 'ok': 'Smaller percentage uncertainty means a more precise result.', 'dp': 2, 'args': [(50.2, 0.3), (5.78, 0.01), (23.7, 0.1), (-4.0, 0.2)]},
}
print('Self-checks ready. Now carry on with Part A.')

## Part A – Explore (about 10 minutes)
### 🔬 Online lab for this unit (free open educational resources)
- **[Vernier Calliper simulation (Open Source Physics @ Singapore)](https://iwant2study.moe.edu.sg/sg.iwant2study.org/ospsg/index.php/interactive-resources/physics/01-foundations-of-physics/1003-vernierccalipermmversion)** – measure objects, try the zero-error slider and check your reading. CC BY-SA licence.
- **[Digital Micrometer simulation (Open Source Physics @ Singapore)](https://iwant2study.moe.edu.sg/sg.iwant2study.org/ospsg/index.php/1189-digitalmicrometer)** – practise reading the sleeve and thimble scales. CC BY licence.
- **[PhET Pendulum Lab](https://phet.colorado.edu/en/simulations/pendulum-lab)** – time oscillations with the built-in stopwatch for the graph task.

Open the online lab in a new tab, try it for a few minutes, then come back here. **Label every reading you take there as "simulated".**

**Reading a vernier.** Reading = main-scale reading (just before the vernier zero) + (number of the vernier mark that lines up) × **least count**. For a scale with 10 vernier divisions over 9 mm, the least count is 0.1 mm.

**Zero error.** Close the jaws. If the vernier zero is to the *right* of the main zero, the zero error is *positive*: subtract it from every reading.

In [ ]:
import numpy as np

def vernier(main_mm, vernier_mark, least_count=0.1, zero_error=0.0):
    return main_mm + vernier_mark * least_count - zero_error

print('Main 31 mm, mark 4 ->', round(vernier(31, 4), 2), 'mm')
print('Same, zero error +0.3 mm ->', round(vernier(31, 4, zero_error=0.3), 2), 'mm')

## Part B – Practise (about 20 minutes)
**B1.** A vernier calliper (least count 0.1 mm) reads 23 mm on the main scale, and vernier mark 7 lines up. What is the reading in mm?

In [ ]:
answer_2_1 = ...

In [ ]:
check('2.1', answer_2_1)

In [ ]:
answer_2_2 = ...   # the same reading corrected for a zero error of +0.2 mm

In [ ]:
check('2.2', answer_2_2)

**B3.** A micrometer (least count 0.01 mm) shows 5.5 mm on the sleeve and 28 on the thimble. Reading = sleeve + thimble × 0.01. Give the reading in mm.

In [ ]:
answer_2_3 = ...

In [ ]:
check('2.3', answer_2_3)

**B4. Percentage uncertainty.** A length is 50.2 ± 0.3 mm. Percentage uncertainty = (0.3 / 50.2) × 100. Give it to 1 decimal place.

In [ ]:
answer_2_4 = ...

In [ ]:
check('2.4', answer_2_4)

**B5. Combining uncertainties.** A card is L = 50.2 ± 0.3 mm long and W = 20.1 ± 0.2 mm wide. For a product A = L × W, the **fractional** uncertainties add: ΔA/A = ΔL/L + ΔW/W.
Find `(A, ΔA)` in mm², each to 1 decimal place.

In [ ]:
L, dL = 50.2, 0.3
W, dW = 20.1, 0.2
A = ...
dA = ...
answer_2_5 = (A, dA)
print(answer_2_5)

In [ ]:
check('2.5', answer_2_5 if ... not in answer_2_5 else ...)

**B6. A value from a graph.** For a simple pendulum T = 2π√(L/g), so **T² = (4π²/g) L**: a graph of T² against L is a straight line through the origin with gradient 4π²/g. Use the illustrative data below. Find the gradient with `np.polyfit(L, T**2, 1)[0]`, then g = 4π² / gradient (1 decimal place).

In [ ]:
L_data = np.array([0.20, 0.40, 0.60, 0.80, 1.00])      # m   (illustrative)
T_data = np.array([0.899, 1.268, 1.556, 1.795, 2.007])  # s   (illustrative)
import matplotlib.pyplot as plt
plt.plot(L_data, T_data**2, 'o'); plt.xlabel('L (m)'); plt.ylabel('T² (s²)'); plt.title('T² against L'); plt.show()

In [ ]:
gradient = ...
answer_2_6 = ...   # g in m/s²

In [ ]:
check('2.6', answer_2_6)

## Part C – Build a tool (about 20 minutes)
Complete `percent_uncertainty(value, uncertainty)` so that it returns the percentage uncertainty rounded to 2 decimal places. Then the cell uses it to compare instruments.

In [ ]:
def percent_uncertainty(value, uncertainty):
    ...      # YOUR CODE

In [ ]:
check('2.7', percent_uncertainty)

In [ ]:
for name, value, unc in [('ruler', 23.5, 0.5), ('vernier', 23.7, 0.1), ('micrometer', 5.78, 0.01)]:
    p = percent_uncertainty(value, unc)
    print(f'{name:10} {value} ± {unc} mm  ->  {p}%')

## Part D – Your own data and reflection (about 10 minutes)
**D1. Prepares you for the Unit 2 assignment.** In the vernier and micrometer simulations, measure three objects each and record the zero error. Or, in the Pendulum Lab, time 10 oscillations for five lengths and repeat the B6 graph with your own data. Label your data "simulated".

In [ ]:
# D1: your own readings
my_L = [...]
my_T = [...]

**D2. Reflect.**
1. Why do we time 10 or 20 oscillations rather than one?
2. Explain the difference between a random error and a systematic error, with one example of each from this lab.
3. Why should an uncertainty usually be given to only 1 or 2 significant figures?

*Type your answer here (double-click this cell to edit it).*

## Finished? Save your work
In Colab: **File → Save a copy in GitHub** → your `phy107-labs` repository → commit message *"Lab 2 completed"*.
Offline: save with **Ctrl + S**, then upload the file to your repository with **Add file → Upload files** when you are online.

Then go back to Moodle and mark the lab as done.